<a href="https://colab.research.google.com/github/DeepFluxion/MACK_data-visualization/blob/main/Mack_Churn_Analisys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Siga os passos abaixo no Google Colab para configurar seu ambiente:
Passo 1: Abrir o Colab e Instalar Pacotes
Execute as seguintes células no seu notebook Colab:

In [1]:
# Célula 1: Instalação de Bibliotecas
# Instala o pacote para acessar datasets da UCI Machine Learning Repository
!pip install ucimlrepo

# Instala o Plotly (geralmente já vem pré-instalado no Colab, mas é bom garantir)
!pip install plotly

Passo 2: Importar Bibliotecas Essenciais
Execute a importação das bibliotecas.

In [2]:
# Célula 2: Importação de Bibliotecas
import pandas as pd
import plotly.express as px
from ucimlrepo import fetch_ucirepo

Passo 3: Carregar o Dataset de Telecom Iraniana
Utilize o script que a fonte sugere para buscar o dataset diretamente no repositório UCI

In [3]:
# Célula 3: Carregar o Dataset
# Fetch dataset ID 563 (Iranian Churn)
iranian_churn = fetch_ucirepo(id=563)

# data (como pandas dataframes)
X = iranian_churn.data.features
y = iranian_churn.data.targets

# Reunir features (X) e target (y) em um único DataFrame para análise
df = pd.concat([X, y], axis=1)

print("Dataset carregado com sucesso. Primeiras 5 linhas:")
print(df.head())

Dataset carregado com sucesso. Primeiras 5 linhas:
   Call  Failure  Complains  Subscription  Length  Charge  Amount  \
0              8          0                    38               0   
1              0          0                    39               0   
2             10          0                    37               0   
3             10          0                    38               0   
4              3          0                    38               0   

   Seconds of Use  Frequency of use  Frequency of SMS  \
0            4370                71                 5   
1             318                 5                 7   
2            2453                60               359   
3            4198                66                 1   
4            2393                58                 2   

   Distinct Called Numbers  Age Group  Tariff Plan  Status  Age  \
0                       17          3            1       1   30   
1                        4          2            1       

# 2. Análise Exploratória de Dados (EDA) Inicial
Nosso primeiro foco é entender a distribuição da variável target (Churn), que é o ponto central do Slide 1 ("O Custo Oculto: Por Que o Churn é Nossa Métrica Financeira Mais Crítica").

O dataset de telecom iraniana contém 3150 instâncias (clientes). A variável Churn é binária (1: churn, 0: não-churn).

Script para EDA 1: Análise de Imbalance (Taxa Bruta de Churn)
Este script irá calcular a taxa bruta de churn e gerar um gráfico de pizza interativo (plotly.pie) para visualizá-la.

In [5]:
# Célula 4: Análise da Taxa Bruta de Churn (Slide 1)

# 1. Preparação dos dados: Mapear 0 e 1 para rótulos legíveis
df['Churn_Label'] = df['Churn'].map({0: 'Não Churn (Retido)', 1: 'Churn (Cancelamento)'})

# 2. Calcular a Contagem e a Proporção
churn_counts = df['Churn_Label'].value_counts().reset_index()
churn_counts.columns = ['Status', 'Total_Clientes']
total_clientes = churn_counts['Total_Clientes'].sum()
churn_rate = churn_counts[churn_counts['Status'] == 'Churn (Cancelamento)']['Total_Clientes'].iloc[0] / total_clientes * 100

print(f"Taxa de Churn Bruta: {churn_rate:.2f}%")

# 3. Geração do Gráfico de Pizza (Plotly)
fig = px.pie(
    churn_counts,
    values='Total_Clientes',
    names='Status',
    title=f'<b>Distribuição de Clientes: Churn vs. Retenção</b><br> (Total de {total_clientes} Clientes)',
    color='Status',
    color_discrete_map={'Churn (Cancelamento)': 'red', 'Não Churn (Retido)': 'green'},
    hole=0.4  # Cria um gráfico de rosca para melhor visualização
)

# Adiciona anotação da taxa de churn no centro do gráfico
fig.update_traces(textinfo='percent+label', pull=[0.05, 0])
fig.add_annotation(text=f'{churn_rate:.2f}%',
                   x=0.5, y=0.5, font_size=20, showarrow=False)

fig.update_layout(
    font_family="Arial",
    title_x=0.5,
    legend_title="Status do Cliente"
)

fig.show()

Taxa de Churn Bruta: 15.71%


3. Comentários para o Slide 1 (Passo 3)
Após gerar o gráfico de pizza (ou rosca), você terá o valor exato da taxa de churn para apresentar aos executivos:
Elemento
Conteúdo
Comentário Executivo (P3)
Gráfico
Gráfico de Pizza/Rosca mostrando a divisão Churn/Não Churn.
Análise da Taxa Bruta: A taxa calculada de churn é [Valor Calculado]%. Com o nosso dataset, verificamos que a perda de clientes está [Comparação: Acima do Ideal de 4%-5% ou Abaixo do Nível Crítico de 7%].
Fato Relevante
O Custo Oculto.
A retenção de clientes é crucial porque o custo de manter um cliente existente é significativamente menor do que o custo de adquirir um novo (pode ser 5 a 7 vezes mais caro). Focar em reduzir este número pode aumentar a lucratividade em até 40%.
Chamada para Ação
Transição.
Precisamos urgentemente entender o perfil desses clientes que estamos perdendo (a taxa de gross attrition) e quais fatores controláveis, como falhas de serviço ou suporte (voluntary churn), estão impulsionando essa evasão.

Próxima Etapa: Preparação do Script para o Slide 2 (Segmentação)
A próxima etapa lógica na EDA é perfilar os churners por atributos demográficos e contratuais, conforme planejado no Slide 2 (Segmentação por Idade e Tipo de Plano).
EDA 2: Perfil do Churner (Age Group e Tariff Plan)
O dataset possui os seguintes atributos relevantes para esta análise de perfil:
• Age Group: Atributo ordinal (1: grupo etário mais jovem, 5: grupo etário mais velho).
• Tariff Plan: Binário (1: Pré-pago/Pay as you go, 2: Contratual).
O script a seguir fará a análise cruzada desses fatores versus o churn.

In [8]:
# Célula 5: Análise Cruzada: Churn por Idade e Plano (Slide 2)

# 1. Mapeamento de variáveis ordinais para leitura executiva
# Mapear Age Group
age_map = {1: '1-Jovem', 2: '2-Médio-Jovem', 3: '3-Adulto', 4: '4-Médio-Velho', 5: '5-Velho'}
df['Age_Group_Label'] = df['Age Group'].map(age_map)

# Mapear Tariff Plan
tariff_map = {1: 'Pré-pago (Pay as you go)', 2: 'Contratual'}
df['Tariff_Plan_Label'] = df['Tariff Plan'].map(tariff_map)

# 2. Calcular a Taxa de Churn por Plano Tarifário
churn_by_tariff = df.groupby('Tariff_Plan_Label')['Churn'].mean().reset_index()
churn_by_tariff['Churn_Rate'] = churn_by_tariff['Churn'] * 100

# 3. Geração do Gráfico de Barras Agrupadas (Plotly)
fig_tariff = px.bar(
    churn_by_tariff,
    x='Tariff_Plan_Label',
    y='Churn_Rate',
    color='Tariff_Plan_Label',
    title='<b>Taxa de Churn por Tipo de Plano Tarifário</b>',
    labels={'Tariff_Plan_Label': 'Tipo de Plano', 'Churn_Rate': 'Taxa de Churn (%)'},
    text='Churn_Rate'
)

fig_tariff.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig_tariff.update_layout(title_x=0.5, showlegend=False)
fig_tariff.show()


# 4. Geração do Gráfico de Churn por Grupo Etário (Box Plot ou Barras)
# Usando Box Plot para visualizar a distribuição do risco vs. Age Group
fig_age = px.box(
    df,
    x='Age_Group_Label',
    y='Churn',
    color='Age_Group_Label',
    title='<b>Risco de Churn (0=Retido, 1=Churn) por Grupo Etário</b>',
    labels={'Age_Group_Label': 'Grupo Etário', 'Churn': 'Status Churn (0/1)'},
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig_age.update_layout(title_x=0.5)
fig_age.show()

In [7]:
# Célula 5 (Parte 2 Revisada): Análise da Taxa de Churn por Grupo Etário

# 1. Mapeamento de variáveis ordinais para leitura executiva (se ainda não tiver sido feito)
# 'Age Group' é ordinal (1: jovem, 5: velho) [2]
age_map = {1: '1 - Jovem', 2: '2 - Médio-Jovem', 3: '3 - Adulto', 4: '4 - Médio-Velho', 5: '5 - Velho'}
df['Age_Group_Label'] = df['Age Group'].map(age_map)

# 2. Calcular a Taxa de Churn por Grupo Etário
# A média da variável binária 'Churn' (onde 1=Churn) é a taxa de Churn
churn_by_age = df.groupby('Age_Group_Label')['Churn'].mean().reset_index()
churn_by_age['Churn_Rate'] = churn_by_age['Churn'] * 100 # Converter para porcentagem

# 3. Geração do Gráfico de Barras (Plotly)
fig_age = px.bar(
    churn_by_age,
    x='Age_Group_Label',
    y='Churn_Rate',
    color='Churn_Rate', # Cor varia pela taxa para dar mais ênfase
    title='<b>Taxa de Churn (%) por Grupo Etário</b>',
    labels={'Age_Group_Label': 'Grupo Etário', 'Churn_Rate': 'Taxa de Churn (%)'},
    color_continuous_scale=px.colors.sequential.Reds_r # Usar uma escala de cores que aumenta com o risco
)

# Adicionar o valor da taxa de churn na barra
fig_age.update_traces(texttemplate='%{y:.2f}%', textposition='outside')

fig_age.update_layout(
    title_x=0.5,
    xaxis={'categoryorder':'array', 'categoryarray': list(age_map.values())}, # Garante a ordem correta das categorias
    coloraxis_showscale=False # Remove a barra de cores lateral
)

fig_age.show()

Sua solicitação é crucial para o storytelling executivo, pois a segmentação mais granular (cruzando Age Group e Tariff Plan) permite que os executivos vejam onde o risco de churn é maior, o que apoia a alocação eficiente de recursos de retenção.
O Slide 2 foi planejado para responder à pergunta: "Quem Está Cancelando? Segmentação por Idade e Tipo de Plano11].
O Slide 2 foi planejado para responder à pergunta: "Quem Está Cancelando? Segmentação por Idade e Tipo de Plano". Vamos criar um script em Python (usando Plotly) para gerar um Gráfico de Barras Agrupadas (ou Múltiplas Barras) que visualize a Taxa de Churn (%) para cada Age Group, dividida por Tariff Plan.
Este tipo de visualização é poderoso, pois permite que os gestores de marketing, por exemplo, comparem se o churn é um problema maior para jovens em planos pré-pagos (onde se pode focar em laços financeiros, como descontos) ou para clientes mais velhos em planos contratuais.
Script Python para Análise Cruzada (Slide 2 - Versão Final)
Ajuste a Célula 5 no seu ambiente Colab para incluir esta análise de risco cruzada.
Preparação do Dataframe e Geração do Gráfico de Barras Agrupadas

In [9]:
# Célula 5 (Final): Análise Cruzada de Churn por Grupo Etário e Plano Tarifário (Slide 2)

# 1. Mapeamento dos Rótulos (se ainda não tiver sido feito)
# Atributos do dataset: Age Group (ordinal 1-5), Tariff Plan (1: Pay as you go, 2: Contratual) [4]
age_map = {1: '1 - Jovem', 2: '2 - Médio-Jovem', 3: '3 - Adulto', 4: '4 - Médio-Velho', 5: '5 - Velho'}
df['Age_Group_Label'] = df['Age Group'].map(age_map)

tariff_map = {1: 'Pré-pago (Pay as you go)', 2: 'Contratual'}
df['Tariff_Plan_Label'] = df['Tariff Plan'].map(tariff_map)

# 2. Calcular a Taxa de Churn Agregada por Grupo Etário E Tipo de Plano
# Agrupamos por ambas as variáveis e calculamos a média do Churn (proporção)
churn_cross_analysis = df.groupby(['Age_Group_Label', 'Tariff_Plan_Label'])['Churn'].mean().reset_index()
churn_cross_analysis['Churn_Rate'] = churn_cross_analysis['Churn'] * 100 # Converter para porcentagem

# 3. Geração do Gráfico de Barras Agrupadas (Plotly Express)
fig_cross = px.bar(
    churn_cross_analysis,
    x='Age_Group_Label',
    y='Churn_Rate',
    color='Tariff_Plan_Label', # Variável de agrupamento (cor)
    barmode='group', # Define que as barras devem ser agrupadas lado a lado
    title='<b>Taxa de Churn (%) por Grupo Etário e Tipo de Plano</b>',
    labels={'Age_Group_Label': 'Grupo Etário', 'Churn_Rate': 'Taxa de Churn (%)', 'Tariff_Plan_Label': 'Tipo de Plano'},
    color_discrete_map={'Pré-pago (Pay as you go)': 'blue', 'Contratual': 'orange'},
    text_auto=True # Adiciona o valor da taxa de churn diretamente acima da barra
)

# Otimização do Layout para a Apresentação Executiva
fig_cross.update_traces(texttemplate='%{y:.1f}%', textposition='outside')
fig_cross.update_layout(
    title_x=0.5,
    xaxis={'categoryorder':'array', 'categoryarray': list(age_map.values())}, # Garante a ordem ordinal correta
    legend_title_text='Tipo de Plano Tarifário'
)

fig_cross.show()

In [11]:
# Importar bibliotecas (se a Célula 2 não foi executada novamente)
import pandas as pd
import plotly.express as px

# Assumindo que o DataFrame 'df' com a coluna 'Churn' já foi carregado
# Se a coluna 'Churn_Label' já existe, ela será sobrescrita.

# 1. Preparação dos dados: Mapear 0 e 1 para rótulos legíveis
# Churn: binário (1: churn, 0: non-churn) - Class label [2]
df['Churn_Label'] = df['Churn'].map({0: 'Não Churn (Retido)', 1: 'Churn (Cancelamento)'})

# 2. Calcular a Contagem e a Proporção
churn_counts = df['Churn_Label'].value_counts().reset_index()
churn_counts.columns = ['Status', 'Total_Clientes']
total_clientes = churn_counts['Total_Clientes'].sum()
# Calcula a taxa de churn para exibição no gráfico
churn_rate_value = df['Churn'].mean() * 100

# 3. Geração do Gráfico de Rosca (Plotly)
fig = px.pie(
    churn_counts,
    values='Total_Clientes',
    names='Status',
    title=f'<b>Distribuição de Clientes: Churn vs. Retenção</b><br> (Base de 12 meses: {total_clientes} Clientes)',
    color='Status',
    # Cores de alto contraste para o impacto executivo
    color_discrete_map={'Churn (Cancelamento)': '#E31937', 'Não Churn (Retido)': '#1F78B4'},
    hole=0.5  # Cria um gráfico de rosca, mais limpo para apresentações
)

# Configurações de Aparência:
fig.update_traces(
    textinfo='percent+value', # Mostra a porcentagem e o valor absoluto
    pull=[0.05 if status == 'Churn (Cancelamento)' else 0 for status in churn_counts['Status']], # Destaca a fatia de Churn
    marker=dict(line=dict(color='#000000', width=1))
)

# Adiciona anotação central com a Taxa de Churn (%)
fig.add_annotation(
    text=f'{churn_rate_value:.2f}%',
    x=0.5, y=0.5, font_size=28, showarrow=False,
    font=dict(color='#E31937') # Cor do texto central em destaque (vermelho)
)

fig.update_layout(
    font_family="Arial",
    title_x=0.5,
    legend_title="Status do Cliente",
    # Altura e largura para melhor ajuste em um slide
    height=500,
    width=700
)

fig.show()

In [12]:
# Célula 5: Análise Cruzada de Churn por Grupo Etário e Plano Tarifário (Slide 2)

# 1. Mapeamento dos Rótulos (para clareza executiva)
# Atributos do dataset: Age Group (ordinal 1-5), Tariff Plan (1: Pay as you go, 2: Contratual) [1]
age_map = {1: '1 - Jovem', 2: '2 - Médio-Jovem', 3: '3 - Adulto', 4: '4 - Médio-Velho', 5: '5 - Velho'}
df['Age_Group_Label'] = df['Age Group'].map(age_map)

tariff_map = {1: 'Pré-pago (Pay as you go)', 2: 'Contratual'}
df['Tariff_Plan_Label'] = df['Tariff Plan'].map(tariff_map)

# 2. Calcular a Taxa de Churn Agregada por Grupo Etário E Tipo de Plano
# Calculamos a média da variável binária 'Churn' agrupada pelos dois fatores
churn_cross_analysis = df.groupby(['Age_Group_Label', 'Tariff_Plan_Label'])['Churn'].mean().reset_index()
churn_cross_analysis['Churn_Rate'] = churn_cross_analysis['Churn'] * 100 # Converter para porcentagem

# 3. Geração do Gráfico de Barras Agrupadas (Plotly Express)
fig_cross = px.bar(
    churn_cross_analysis,
    x='Age_Group_Label',
    y='Churn_Rate',
    color='Tariff_Plan_Label', # Variável de agrupamento (cor)
    barmode='group', # Barras agrupadas lado a lado
    title='<b>Risco de Churn (%) Segmentado por Idade e Plano Tarifário</b>',
    labels={'Age_Group_Label': 'Grupo Etário', 'Churn_Rate': 'Taxa de Churn (%)', 'Tariff_Plan_Label': 'Tipo de Plano'},
    color_discrete_map={'Pré-pago (Pay as you go)': '#4C78A8', 'Contratual': '#F58518'}, # Cores distintas
    text_auto=True
)

# Otimização do Layout
fig_cross.update_traces(texttemplate='%{y:.1f}%', textposition='outside')
fig_cross.update_layout(
    font_family="Arial",
    title_x=0.5,
    xaxis={'categoryorder':'array', 'categoryarray': list(age_map.values())}, # Mantém a ordem ordinal
    legend_title_text='Tipo de Plano Tarifário',
    height=550,
    width=800
)

fig_cross.show()

In [16]:
# Célula 6: Análise da Média de Falhas e Reclamações por Status de Churn (Slide 3)

# ATENÇÃO: Verifique o nome real da coluna em df.columns.
# Se for 'Call_Failures', substitua 'Call Failures' por 'Call_Failures' abaixo.
COLUNAS_ANALISE = ['Call  Failure', 'Complains']

# 1. Agregação dos dados: calcular a média das métricas para Churn vs. Não Churn
# Churn_Label (rótulo criado na Célula 4)
churn_metrics = df.groupby('Churn_Label')[COLUNAS_ANALISE].mean().reset_index()

# 2. Renomear e converter Complains
# 'Complains' é binário (0: No complaint, 1: complaint) [1]. A média é a taxa de reclamação.
churn_metrics = churn_metrics.rename(columns={'Complains': 'Taxa de Reclamação (%)'})
churn_metrics['Taxa de Reclamação (%)'] = churn_metrics['Taxa de Reclamação (%)'] * 100

# 3. Reformatar (Melt) para Plotly: Transformar de wide para long format
df_plot_qualidade = churn_metrics.melt(
    id_vars='Churn_Label',
    var_name='Métrica de Qualidade',
    value_name='Valor Médio'
)

# Ajuste do nome da métrica Call Failures para clareza no gráfico
df_plot_qualidade.loc[df_plot_qualidade['Métrica de Qualidade'] == 'Call Failures', 'Métrica de Qualidade'] = 'Média de Falhas de Chamada'


# 4. Geração do Gráfico de Barras Agrupadas (Plotly Express)
fig_qualidade = px.bar(
    df_plot_qualidade,
    x='Métrica de Qualidade',
    y='Valor Médio',
    color='Churn_Label',
    barmode='group',
    title='<b>Impacto da Qualidade de Serviço no Risco de Churn</b>',
    labels={'Métrica de Qualidade': 'Indicador Operacional', 'Valor Médio': 'Valor Médio (Unidade/%)', 'Churn_Label': 'Status do Cliente'},
    color_discrete_map={'Churn (Cancelamento)': '#E31937', 'Não Churn (Retido)': '#1F78B4'},
)

# Otimização do Layout
fig_qualidade.update_traces(texttemplate='%{y:.1f}', textposition='outside')
fig_qualidade.update_layout(
    font_family="Arial",
    title_x=0.5,
    legend_title_text='Status',
    yaxis_title='Valor Médio (Reclamação em % / Falhas em Unidade)',
    height=550,
    width=800
)

fig_qualidade.show()

In [17]:
# Variáveis contínuas de uso
VARS_USO = ['Frequency of use', 'Seconds of Use']

# Mapeamento para títulos de gráficos mais claros
titles_map = {
    'Frequency of use': 'Frequência de Uso (Total de Chamadas)',
    'Seconds of Use': 'Segundos de Uso (Total de Chamadas)'
}

# Cores e rótulos (Consistentes com slides anteriores)
color_map = {'Churn (Cancelamento)': '#E31937', 'Não Churn (Retido)': '#1F78B4'}

for var in VARS_USO:
    # Criação do Box Plot para cada variável de uso
    fig_box = px.box(
        df,
        x='Churn_Label',
        y=var,
        color='Churn_Label',
        title=f'<b>Distribuição de {titles_map[var]} por Status de Churn</b>',
        labels={'Churn_Label': 'Status do Cliente', var: titles_map[var]},
        color_discrete_map=color_map
    )

    # Otimização do Layout
    fig_box.update_layout(
        font_family="Arial",
        title_x=0.5,
        legend_title_text='Status',
        height=550,
        width=800
    )

    fig_box.show()

In [18]:
# Célula 8: Análise da Distribuição do Valor do Cliente (Customer Value) no Churn (Slide 5)

import pandas as pd
import plotly.express as px

# 1. Filtrar o DataFrame apenas para os clientes que fizeram Churn (Churn_Label == 'Churn (Cancelamento)')
churners_df = df[df['Churn_Label'] == 'Churn (Cancelamento)'].copy()

# 2. Verificar se há dados de Customer Value disponíveis para os churners
if churners_df.empty:
    print("Nenhum cliente com Churn encontrado no DataFrame para análise.")
else:
    # 3. Geração do Histograma do Customer Value para os Churners (Plotly Express)
    fig_value = px.histogram(
        churners_df,
        x='Customer Value',
        nbins=20, # Número de caixas (bins) para detalhar a distribuição
        title='<b>Distribuição do Valor Calculado do Cliente (Customer Value) entre Churners</b>',
        labels={'Customer Value': 'Valor Calculado do Cliente', 'count': 'Número de Clientes Perdidos'},
        color_discrete_sequence=['#E31937'] # Cor vermelha, consistente com o tema Churn
    )

    # Adicionar uma linha vertical para a Média do Customer Value dos Churners
    mean_customer_value = churners_df['Customer Value'].mean()
    fig_value.add_vline(
        x=mean_customer_value,
        line_dash="dash",
        line_color="black",
        annotation_text=f"Média de Valor Perdido: {mean_customer_value:.2f}",
        annotation_position="top right"
    )

    # Otimização do Layout
    fig_value.update_layout(
        font_family="Arial",
        title_x=0.5,
        xaxis_title="Valor Calculado do Cliente (Customer Value)",
        yaxis_title="Frequência (Clientes)",
        height=550,
        width=800
    )

    fig_value.show()

In [20]:
# Célula 9 CORRIGIDA: Probabilidade de Churn por Duração da Assinatura (Subscription Length) (Slide 6)

import pandas as pd
import plotly.express as px

# CORREÇÃO: Usamos o nome da coluna com os dois espaços ('  ') conforme listado na sua mensagem de erro.
SUBSCRIPTION_VAR = 'Subscription  Length'

# Variável de interesse: Subscription Length (Duração total em meses)
# SUBSCRIPTION_VAR = 'Subscription Length' # Versão anterior com erro

# Mapeamento e Cores (Assumindo que Churn_Label foi criado na Célula 4)
color_map = {'Churn (Cancelamento)': '#E31937', 'Não Churn (Retido)': '#1F78B4'}


# 1. Geração do Histograma/Densidade Comparativa
fig_density = px.histogram(
    df,
    x=SUBSCRIPTION_VAR,
    color='Churn_Label',
    histnorm='percent',
    marginal='box',
    opacity=0.7,
    barmode='overlay',
    title='<b>Distribuição da Duração da Assinatura por Status de Churn</b>',
    labels={'Churn_Label': 'Status do Cliente', SUBSCRIPTION_VAR: 'Duração da Assinatura (Meses)', 'percent': 'Porcentagem (%)'},
    color_discrete_map=color_map
)

# Otimização do Layout
fig_density.update_layout(
    font_family="Arial",
    title_x=0.5,
    legend_title_text='Status',
    yaxis_title="Percentual de Clientes (%)",
    height=600,
    width=900
)

fig_density.show()


# 2. Gráfico de Linha (Taxa de Churn ao longo do tempo)
# Calcula a taxa média de churn para cada mês de contrato presente no dataset
churn_rate_by_length = df.groupby(SUBSCRIPTION_VAR)['Churn'].mean().reset_index()
churn_rate_by_length['Churn_Rate_Pct'] = churn_rate_by_length['Churn'] * 100

fig_line = px.line(
    churn_rate_by_length,
    x=SUBSCRIPTION_VAR,
    y='Churn_Rate_Pct',
    title='<b>Taxa de Churn (%) em Relação à Duração da Assinatura</b>',
    labels={SUBSCRIPTION_VAR: 'Duração da Assinatura (Meses)', 'Churn_Rate_Pct': 'Taxa de Churn (%)'},
    line_shape='spline',
    markers=True
)

fig_line.update_layout(
    font_family="Arial",
    title_x=0.5,
    yaxis_title="Taxa de Churn (%)"
)
fig_line.show()

In [31]:
# Célula 10 Revisada: Gráfico de Dispersão/Bolhas para Priorização Preditiva (Slide 7)

import pandas as pd
import plotly.express as px

# Variáveis de Risco e Valor (Nomes corrigidos com base no seu DataFrame)
X_RISK = 'Call  Failure'
Y_RISK = 'Frequency of use'
SIZE_VALUE = 'Customer Value'

# Rótulos e Cores
color_map = {'Churn (Cancelamento)': '#E31937', 'Não Churn (Retido)': '#1F78B4'}

# Cálculo dos Limiares Críticos para demarcação
# Limiar X (Atrito Operacional - Call Failures): 75º Percentil (Clientes com alto número de falhas)
limiar_call_failure = df[X_RISK].quantile(0.5)

# Limiar Y (Engajamento - Frequency of use): 25º Percentil (Clientes com baixo número de uso)
limiar_frequency_use = df[Y_RISK].quantile(0.5)


# 1. Geração do Gráfico de Bolhas (Scatter Plot)
fig_scatter = px.scatter(
    df,
    x=X_RISK,
    y=Y_RISK,
    size=SIZE_VALUE,
    color='Churn_Label',
    hover_data=[SIZE_VALUE, X_RISK, Y_RISK],
    title='<b>SLIDE 7: Priorização de Clientes em Risco e Alto Valor</b>',
    labels={
        X_RISK: 'Fator de Atrito (Call Failures - Maior para a Direita)',
        Y_RISK: 'Fator Comportamental (Frequency of use - Maior para Cima)',
        'Churn_Label': 'Status Histórico',
        SIZE_VALUE: 'Valor Calculado (Customer Value)'
    },
    color_discrete_map=color_map,
    opacity=0.6
)

# 2. ADIÇÃO DOS LIMIARES (Linhas Pontilhadas)

# Linha Vertical para Call Failures (Atrito)
fig_scatter.add_vline(
    x=limiar_call_failure,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Limite de Alto Atrito: {limiar_call_failure:.0f} Falhas",
    annotation_position="top right"
)

# Linha Horizontal para Frequency of use (Engajamento)
fig_scatter.add_hline(
    y=limiar_frequency_use,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Limite de Baixo Uso: {limiar_frequency_use:.0f} Chamadas",
    annotation_position="bottom left"
)

# 3. Otimização do Layout
fig_scatter.update_layout(
    font_family="Arial",
    title_x=0.5,
    legend_title_text='Status',
    # Remove a anotação antiga para evitar confusão com os limiares
    annotations=[],
    height=600,
    width=900
)

# Destaque visual da Área Crítica (Quadrante Inferior Direito)
# Este é o quadrante onde Call Failures é ALTO e Frequency of use é BAIXO.
# Neste quadrante é onde as bolhas grandes (alto Customer Value) representam a prioridade de retenção.

# Adicionar uma anotação para a área de alto risco
fig_scatter.add_annotation(
    x=df[X_RISK].max(), y=df[Y_RISK].min(),
    text="ÁREA DE INTERVENÇÃO PROATIVA (Alto Risco e Baixo Engajamento)",
    showarrow=False,
    xanchor="right", yanchor="bottom",
    font=dict(size=12, color="#E31937"),
    bgcolor="rgba(255, 0, 0, 0.1)",
    bordercolor="#E31937", borderwidth=1
)


fig_scatter.show()

In [33]:
# Célula 12: Gráfico de Priorização com Quadrantes pela Mediana e Histogramas Marginais

import pandas as pd
import plotly.express as px

# Variáveis de Risco e Valor (Nomes corrigidos)
X_RISK = 'Call  Failure'
Y_RISK = 'Frequency of use'
SIZE_VALUE = 'Customer Value'

# Rótulos e Cores
color_map = {'Churn (Cancelamento)': '#E31937', 'Não Churn (Retido)': '#1F78B4'}

# Cálculo dos Limiares (Medianas) para demarcação dos quadrantes
# A mediana divide o dataset em 50%
mediana_call_failure = df[X_RISK].median()
mediana_frequency_use = df[Y_RISK].median()


# 1. Geração do Gráfico de Bolhas com Histogramas Marginais
fig_scatter_marginal = px.scatter(
    df,
    x=X_RISK,
    y=Y_RISK,
    size=SIZE_VALUE,
    color='Churn_Label',
    hover_data=[SIZE_VALUE, X_RISK, Y_RISK],
    title='<b>Priorização de Risco por Valor com Análise de Distribuição</b>',
    labels={
        X_RISK: 'Fator de Atrito (Call Failures)',
        Y_RISK: 'Fator Comportamental (Frequency of use)',
        'Churn_Label': 'Status Histórico',
        SIZE_VALUE: 'Valor Calculado'
    },
    color_discrete_map=color_map,
    opacity=0.6,
    # Adicionando Histogramas Marginais
    marginal_x='histogram',
    marginal_y='histogram'
)

# 2. ADIÇÃO DOS LIMIARES (Linhas Pontilhadas na Mediana)

# Linha Vertical para Call Failures (Mediana)
fig_scatter_marginal.add_vline(
    x=mediana_call_failure,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Mediana Atrito: {mediana_call_failure:.0f}",
    annotation_position="top right"
)

# Linha Horizontal para Frequency of use (Mediana)
fig_scatter_marginal.add_hline(
    y=mediana_frequency_use,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Mediana Uso: {mediana_frequency_use:.0f}",
    annotation_position="bottom left"
)

# 3. Destaque Visual para o Quadrante Crítico (Baixo Uso / Alto Atrito)
fig_scatter_marginal.add_annotation(
    x=df[X_RISK].max(), y=df[Y_RISK].min(),
    text="ALTA PRIORIDADE DE INTERVENÇÃO",
    showarrow=False,
    xanchor="right", yanchor="bottom",
    font=dict(size=12, color="#E31937", weight="bold"),
    bgcolor="rgba(255, 0, 0, 0.1)",
    bordercolor="#E31937", borderwidth=1
)

# Otimização do Layout
fig_scatter_marginal.update_layout(
    font_family="Arial",
    title_x=0.5,
    legend_title_text='Status',
    height=750, # Aumenta a altura para acomodar os histogramas
    width=1000
)

fig_scatter_marginal.show()

In [35]:
# Célula 12 CORRIGIDA: Subplot da Distribuição do Valor do Cliente (Customer Value) por Quadrante

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Variáveis e Limiares (Medianas)
X_RISK = 'Call  Failure' # Coluna Corrigida
Y_RISK = 'Frequency of use'
SIZE_VALUE = 'Customer Value'

# Cálculo dos Limiares (Medianas)
mediana_cf = df[X_RISK].median()
mediana_fu = df[Y_RISK].median()

# 1. Classificação dos Clientes nos Quadrantes (Simplificada)
# Os rótulos serão usados para títulos e filtros, então devem ser consistentes.

# Padrão: Baixo Atrito (< Mediana) e Alto Uso (>= Mediana)
df['Quadrant_Key_Title'] = 'Q4: Baixo Atrito / Alto Uso'

# Q1: ALTO RISCO (Alto Atrito / Baixo Uso) - PRIORIDADE MÁXIMA
df.loc[(df[X_RISK] >= mediana_cf) & (df[Y_RISK] < mediana_fu), 'Quadrant_Key_Title'] = 'Q1: ALTO RISCO (PRIORIDADE)'

# Q2: Alto Atrito (>= Mediana) e Alto Uso (>= Mediana)
df.loc[(df[X_RISK] >= mediana_cf) & (df[Y_RISK] >= mediana_fu), 'Quadrant_Key_Title'] = 'Q2: Alto Atrito / Alto Uso'

# Q3: Baixo Atrito (< Mediana) e Baixo Uso (< Mediana)
df.loc[(df[X_RISK] < mediana_cf) & (df[Y_RISK] < mediana_fu), 'Quadrant_Key_Title'] = 'Q3: Baixo Atrito / Baixo Uso'


# 2. Cálculo das Estatísticas (Média e Mediana) do Customer Value por Quadrante
stats_df = df.groupby('Quadrant_Key_Title')[SIZE_VALUE].agg(['mean', 'median']).reset_index()
stats_map = stats_df.set_index('Quadrant_Key_Title').to_dict('index')


# 3. Configuração do Subplot 2x2
titles = [
    'Q4: Baixo Atrito / Alto Uso',           # R(1), C(1)
    'Q2: Alto Atrito / Alto Uso',            # R(1), C(2)
    'Q3: Baixo Atrito / Baixo Uso',          # R(2), C(1)
    'Q1: ALTO RISCO (PRIORIDADE)'          # R(2), C(2)
]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=titles,
    vertical_spacing=0.15,
)

# Mapeamento de cor e posição para o loop
color_priority = {
    'Q1: ALTO RISCO (PRIORIDADE)': '#E31937',
    'Q2: Alto Atrito / Alto Uso': '#FFCC66',
    'Q3: Baixo Atrito / Baixo Uso': '#FFCC66',
    'Q4: Baixo Atrito / Alto Uso': '#1F78B4'
}
position_map = {title: (i // 2 + 1, i % 2 + 1) for i, title in enumerate(titles)}


# 4. Geração dos Histograms e Adição de Linhas (Média e Mediana)
for title in titles:
    r, c = position_map[title]

    # Filtra os dados para o quadrante
    q_data = df[df['Quadrant_Key_Title'] == title]

    # Extrai estatísticas
    q_mean = stats_map[title]['mean']
    q_median = stats_map[title]['median']

    # Histograma
    fig.add_trace(
        go.Histogram(
            x=q_data[SIZE_VALUE],
            name=title,
            marker_color=color_priority[title],
            hovertemplate = 'Valor: %{x}<br>Contagem: %{y}<extra></extra>'
        ),
        row=r, col=c
    )

    # Adicionando Linhas e Anotações para Média e Mediana
    # A referência Y para o subplot (yref) é crítica
    y_ref_index = 2 * (r - 1) + c

    # Linha para Média (Azul, tracejada)
    fig.add_shape(
        type="line", x0=q_mean, x1=q_mean, y0=0, y1=1, yref="y"+str(y_ref_index),
        line=dict(color="blue", width=2, dash="dash"),
        row=r, col=c
    )
    fig.add_annotation(
        x=q_mean, y=0.9, yref="y"+str(y_ref_index), xanchor='left',
        text=f"Média: {q_mean:.2f}",
        showarrow=False, font=dict(color="blue", size=10),
        row=r, col=c
    )

    # Linha para Mediana (Verde, pontilhada)
    fig.add_shape(
        type="line", x0=q_median, x1=q_median, y0=0, y1=1, yref="y"+str(y_ref_index),
        line=dict(color="green", width=2, dash="dot"),
        row=r, col=c
    )
    fig.add_annotation(
        x=q_median, y=0.8, yref="y"+str(y_ref_index), xanchor='right',
        text=f"Mediana: {q_median:.2f}",
        showarrow=False, font=dict(color="green", size=10),
        row=r, col=c
    )

# 5. Otimização Final
fig.update_layout(
    title_text="<b>SLIDE 8: Distribuição do Valor do Cliente (Customer Value) por Segmento de Risco</b>",
    title_x=0.5,
    height=750,
    width=1100,
    showlegend=False
)

# Ajuste dos Títulos dos Eixos
fig.update_yaxes(title_text="Contagem de Clientes", row=1, col=1)
fig.update_yaxes(title_text="Contagem de Clientes", row=2, col=1)
fig.update_xaxes(title_text="Valor do Cliente (Customer Value)", row=2, col=1)
fig.update_xaxes(title_text="Valor do Cliente (Customer Value)", row=2, col=2)

fig.show()

In [38]:
# Célula 12 CORRIGIDA: Subplot da Distribuição do Valor do Cliente (Customer Value) por Quadrante com Eixos Y Independentes

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Variáveis e Limiares (Medianas)
X_RISK = 'Call  Failure'
Y_RISK = 'Frequency of use'
SIZE_VALUE = 'Customer Value'

# Cálculo dos Limiares (Medianas)
mediana_cf = df[X_RISK].median()
mediana_fu = df[Y_RISK].median()

# 1. Classificação dos Clientes nos Quadrantes
df['Quadrant_Key_Title'] = 'Q4: Baixo Atrito / Alto Uso'
df.loc[(df[X_RISK] >= mediana_cf) & (df[Y_RISK] < mediana_fu), 'Quadrant_Key_Title'] = 'Q1: ALTO RISCO (PRIORIDADE)'
df.loc[(df[X_RISK] >= mediana_cf) & (df[Y_RISK] >= mediana_fu), 'Quadrant_Key_Title'] = 'Q2: Alto Atrito / Alto Uso'
df.loc[(df[X_RISK] < mediana_cf) & (df[Y_RISK] < mediana_fu), 'Quadrant_Key_Title'] = 'Q3: Baixo Atrito / Baixo Uso'

# 2. Cálculo das Estatísticas (Média e Mediana)
stats_df = df.groupby('Quadrant_Key_Title')[SIZE_VALUE].agg(['mean', 'median']).reset_index()
stats_map = stats_df.set_index('Quadrant_Key_Title').to_dict('index')

# 3. Configuração do Subplot 2x2 com EIXOS Y INDEPENDENTES
titles = [
    'Q4: Baixo Atrito / Alto Uso',
    'Q2: Alto Atrito / Alto Uso',
    'Q3: Baixo Atrito / Baixo Uso',
    'Q1: ALTO RISCO (PRIORIDADE)'
]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=titles,
    vertical_spacing=0.15,
    shared_yaxes=False
)

color_priority = {
    'Q1: ALTO RISCO (PRIORIDADE)': '#E31937',
    'Q2: Alto Atrito / Alto Uso': '#FFCC66',
    'Q3: Baixo Atrito / Baixo Uso': '#FFCC66',
    'Q4: Baixo Atrito / Alto Uso': '#1F78B4'
}
position_map = {title: (i // 2 + 1, i % 2 + 1) for i, title in enumerate(titles)}


# 4. Geração dos Histograms (com Densidade) e Adição de Linhas Pontilhadas
for title in titles:
    r, c = position_map[title]

    q_data = df[df['Quadrant_Key_Title'] == title]
    q_mean = stats_map[title]['mean']
    q_median = stats_map[title]['median']
    y_ref_index = 2 * (r - 1) + c

    # Determinação da referência correta do eixo Y
    if y_ref_index == 1:
        # Para o primeiro eixo (y1), a referência é 'y'
        y_ref = "y"
        y_ref_domain = "y domain"
    else:
        # Para os eixos subsequentes (y2, y3, y4...), a referência é 'yN'
        y_ref = "y" + str(y_ref_index)
        y_ref_domain = "y" + str(y_ref_index) + " domain"

    # Histograma (Usando 'probability density' para simular a curva de densidade)
    fig.add_trace(
        go.Histogram(
            x=q_data[SIZE_VALUE],
            name=title,
            marker_color=color_priority[title],
            hovertemplate = 'Valor: %{x}<br>Densidade: %{y}<extra></extra>',
            histnorm='probability density',
            nbinsx=20,
            opacity=0.7
        ),
        row=r, col=c
    )

    # Linha VERTICAL PONTILHADA para Média (Azul)
    fig.add_shape(
        type="line", x0=q_mean, x1=q_mean, y0=0, y1=1,
        yref=y_ref_domain, # Usando a referência corrigida
        line=dict(color="blue", width=2, dash="dash"),
        row=r, col=c
    )
    fig.add_annotation(
        x=q_mean, y=0.9, yref=y_ref_domain, xanchor='left', # Usando a referência corrigida para anotação
        text=f"Média: {q_mean:.2f}",
        showarrow=False, font=dict(color="blue", size=10),
        row=r, col=c
    )

    # Linha VERTICAL PONTILHADA/TRAÇADA para Mediana (Verde)
    fig.add_shape(
        type="line", x0=q_median, x1=q_median, y0=0, y1=1,
        yref=y_ref_domain, # Usando a referência corrigida
        line=dict(color="green", width=2, dash="dot"),
        row=r, col=c
    )
    fig.add_annotation(
        x=q_median, y=0.8, yref=y_ref_domain, xanchor='right', # Usando a referência corrigida para anotação
        text=f"Mediana: {q_median:.2f}",
        showarrow=False, font=dict(color="green", size=10),
        row=r, col=c
    )

# 5. Otimização Final
fig.update_layout(
    title_text="<b>SLIDE 8: Distribuição de Densidade do Valor do Cliente por Segmento de Risco</b>",
    title_x=0.5,
    height=750,
    width=1100,
    showlegend=False
)

# Ajuste dos Títulos dos Eixos
fig.update_yaxes(title_text="Densidade de Probabilidade", row=1, col=1)
fig.update_yaxes(title_text="Densidade de Probabilidade", row=2, col=1)
fig.update_xaxes(title_text="Valor do Cliente (Customer Value)", row=2, col=1)
fig.update_xaxes(title_text="Valor do Cliente (Customer Value)", row=2, col=2)

fig.show()

In [40]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Variáveis e Limiares (Medianas)
X_RISK = 'Call  Failure'
Y_RISK = 'Frequency of use'
SIZE_VALUE = 'Customer Value'

# Cálculo dos Limiares (Medianas)
mediana_cf = df[X_RISK].median()
mediana_fu = df[Y_RISK].median()

# 1. Classificação dos Clientes nos Quadrantes
df['Quadrant_Key_Title'] = 'Q4: Baixo Atrito / Alto Uso'
df.loc[(df[X_RISK] >= mediana_cf) & (df[Y_RISK] < mediana_fu), 'Quadrant_Key_Title'] = 'Q1: ALTO RISCO (PRIORIDADE)'
df.loc[(df[X_RISK] >= mediana_cf) & (df[Y_RISK] >= mediana_fu), 'Quadrant_Key_Title'] = 'Q2: Alto Atrito / Alto Uso'
df.loc[(df[X_RISK] < mediana_cf) & (df[Y_RISK] < mediana_fu), 'Quadrant_Key_Title'] = 'Q3: Baixo Atrito / Baixo Uso'

# 2. Cálculo das Estatísticas (Média, Mediana, SOMA e Percentis)
stats_df = df.groupby('Quadrant_Key_Title')[SIZE_VALUE].agg(
    mean='mean',
    median='median',
    total_value='sum',
    p2_5=lambda x: x.quantile(0.025), # 2.5º percentil
    p97_5=lambda x: x.quantile(0.975) # 97.5º percentil
).reset_index()
stats_map = stats_df.set_index('Quadrant_Key_Title').to_dict('index')

# 3. Configuração do Subplot 2x2 com EIXOS Y INDEPENDENTES
titles = [
    'Q4: Baixo Atrito / Alto Uso',
    'Q2: Alto Atrito / Alto Uso',
    'Q3: Baixo Atrito / Baixo Uso',
    'Q1: ALTO RISCO (PRIORIDADE)'
]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=titles,
    vertical_spacing=0.15,
    shared_yaxes=False
)

color_priority = {
    'Q1: ALTO RISCO (PRIORIDADE)': '#E31937',
    'Q2: Alto Atrito / Alto Uso': '#FFCC66',
    'Q3: Baixo Atrito / Baixo Uso': '#FFCC66',
    'Q4: Baixo Atrito / Alto Uso': '#1F78B4'
}
position_map = {title: (i // 2 + 1, i % 2 + 1) for i, title in enumerate(titles)}

# 4. Geração dos Histograms e Adição de Estatísticas
for title in titles:
    r, c = position_map[title]

    q_data = df[df['Quadrant_Key_Title'] == title]
    stats = stats_map[title]

    y_ref_index = 2 * (r - 1) + c

    # Determinação da referência correta do eixo Y
    if y_ref_index == 1:
        y_ref_domain = "y domain"
    else:
        y_ref_domain = "y" + str(y_ref_index) + " domain"

    # Histograma (REMOVIDO histnorm='probability density' para usar Contagem)
    fig.add_trace(
        go.Histogram(
            x=q_data[SIZE_VALUE],
            name=title,
            marker_color=color_priority[title],
            # Atualiza o hover para mostrar a Contagem
            hovertemplate = 'Valor: %{x}<br>Contagem: %{y}<extra></extra>',
            nbinsx=20,
            opacity=0.7
        ),
        row=r, col=c
    )

    # FAIXA DE VALOR DE CLIENTE COM 95% DE PROBABILIDADE (vrect)
    # y1=1000 (ou um valor alto) é necessário para garantir que a área sombreada cubra as barras
    # Usaremos um valor arbitrariamente alto (0-1 domain) para a altura do vrect
    fig.add_vrect(
        x0=stats['p2_5'], x1=stats['p97_5'],
        fillcolor=color_priority[title],
        opacity=0.1,
        line_width=0,
        row=r, col=c
    )

    # Ajustando o texto do 95% CI para não ficar muito baixo, usando a referência do domínio
    fig.add_annotation(
        text="95% CI",
        x=(stats['p2_5'] + stats['p97_5']) / 2, # Posição no centro da faixa
        y=0.05, yref=y_ref_domain, # Posição baixa no domínio
        showarrow=False, font=dict(size=9),
        row=r, col=c
    )

    # Linha VERTICAL PONTILHADA para Média (Azul)
    fig.add_shape(
        type="line", x0=stats['mean'], x1=stats['mean'], y0=0, y1=1,
        yref=y_ref_domain,
        line=dict(color="blue", width=2, dash="dash"),
        row=r, col=c
    )
    fig.add_annotation(
        x=stats['mean'], y=0.9, yref=y_ref_domain, xanchor='left',
        text=f"Média: {stats['mean']:.0f}",
        showarrow=False, font=dict(color="blue", size=10),
        row=r, col=c
    )

    # Linha VERTICAL PONTILHADA/TRAÇADA para Mediana (Verde)
    fig.add_shape(
        type="line", x0=stats['median'], x1=stats['median'], y0=0, y1=1,
        yref=y_ref_domain,
        line=dict(color="green", width=2, dash="dot"),
        row=r, col=c
    )
    fig.add_annotation(
        x=stats['median'], y=0.8, yref=y_ref_domain, xanchor='right',
        text=f"Mediana: {stats['median']:.0f}",
        showarrow=False, font=dict(color="green", size=10),
        row=r, col=c
    )

    # TOTAL DE VALOR DE CLIENTE COMO ANOTAÇÃO
    fig.add_annotation(
        text=f"<b>Valor Total: R$ {stats['total_value']:.0f}</b>",
        xref="x" + str(y_ref_index), yref="y" + str(y_ref_index),
        x=0.5, y=1.05, # Posição logo acima do plot
        showarrow=False,
        font=dict(size=11, color="black"),
        row=r, col=c
    )

# 5. Otimização Final
fig.update_layout(
    title_text="<b>SLIDE 8: Análise Financeira do Risco - Contagem e Valor do Cliente por Segmento</b>",
    title_x=0.5,
    height=800,
    width=1100,
    showlegend=False
)

# Ajuste dos Títulos dos Eixos para Contagem
fig.update_yaxes(title_text="Contagem de Clientes", row=1, col=1)
fig.update_yaxes(title_text="Contagem de Clientes", row=2, col=1)
fig.update_xaxes(title_text="Valor do Cliente (Customer Value)", row=2, col=1)
fig.update_xaxes(title_text="Valor do Cliente (Customer Value)", row=2, col=2)

fig.show()